# Utility Score Analysis
This notebook verifies the new **exponential decay** and **smoothed concentration bonus** logic across all 6 retail trip types.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast

BASE_DIR = r'C:\Users\sgabalog\Documents\P3\Model\Utility'
TEST_DIR = os.path.join(BASE_DIR, 'testing')
RETAIL_FILE = os.path.join(BASE_DIR, 'retail_centres_processed.parquet')

TRIP_TYPES = ['bulk', 'convenience', 'comparison', 'entertainment', 'food_drink', 'service']

def load_utility(trip_type):
    path = os.path.join(TEST_DIR, f'utility_scores_{trip_type}.parquet')
    return pd.read_parquet(path)

## 1. Load Data
We load all utility datasets and the retail centre baseline data.

In [ ]:
df_rc = pd.read_parquet(RETAIL_FILE)
df_rc['RC_ID'] = df_rc['RC_ID'].astype(str)
df_rc_meta = df_rc.set_index('RC_ID')[['Total_POI_', 'Classifica']]

utilities = {tt: load_utility(tt) for tt in TRIP_TYPES}
print(f"Loaded {len(utilities)} trip types.")

## 2. Distance Decay Verification
We extract travel times and utility scores for a sample agent to check the exponential decay curves.

In [ ]:
def analyze_agent_decay(trip_type, agent_idx=0, mode='drive'):
    df = utilities[trip_type]
    agent = df.iloc[agent_idx]
    
    # Get base trip file to extract travel times (needed for verification)
    # Note: Utility scores are already result of (T * Amenity)
    # We'll plot Score vs. Travel Time
    
    score_cols = [c for c in df.columns if c.endswith(f'_{mode}')]
    scores = agent[score_cols]
    
    # Extract durations from original consumer file (sample batch)
    # For simplicity, we'll map scores back to the RC metadata
    rc_ids = [c.split('_')[0] for c in score_cols]
    res_df = pd.DataFrame({'RC_ID': rc_ids, 'Score': scores.values})
    return res_df

# Sample Plot for Comparison
comp_scores = analyze_agent_decay('comparison')
plt.figure(figsize=(10, 5))
plt.hist(comp_scores['Score'], bins=50)
plt.title('Distribution of Utilities (Agent 0, Comparison, Drive)')
plt.xlabel('Utility Score [0-1]')
plt.ylabel('Count of Centres')
plt.show()

## 3. Concentration vs. Utility
We investigate if larger centres always 'win' or if the smoothing logic correctly differentiates specialized clusters.

In [ ]:
def plot_utility_vs_size(trip_type):
    df = utilities[trip_type]
    # Mean utility for each centre across all agents
    score_cols = [c for c in df.columns if '_drive' in c]
    mean_scores = df[score_cols].mean()
    
    res = pd.DataFrame({
        'RC_ID': [c.replace('_drive', '') for c in mean_scores.index],
        'Mean_Utility': mean_scores.values
    }).set_index('RC_ID')
    
    plot_df = res.join(df_rc_meta)
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=plot_df, x='Total_POI_', y='Mean_Utility', hue='Classifica', alpha=0.6)
    plt.xscale('log')
    plt.title(f'Mean Utility vs Centre Size ({trip_type})')
    plt.show()

plot_utility_vs_size('comparison')
plot_utility_vs_size('convenience')

## 4. Top Destinations Cross-Trip Analysis
Comparing top destinations for a single agent across different trip types.

In [ ]:
agent_hh = utilities['bulk'].index[0]
top_n = 5

print(f"Analyzing Top {top_n} Destinations for HH: {agent_hh}\n")
for tt in TRIP_TYPES:
    df = utilities[tt]
    agent = df.loc[agent_hh]
    score_cols = [c for c in df.columns if '_drive' in c]
    top_rcs = agent[score_cols].sort_values(ascending=False).head(top_n)
    
    print(f"--- {tt.upper()} ---")
    for rc_col, score in top_rcs.items():
        rc_id = rc_col.replace('_drive', '')
        meta = df_rc_meta.loc[rc_id]
        print(f"  RC {rc_id} ({meta['Classifica']}): Utility {score:.3f}")
    print()